# Laboratorium 1: Dekoratory, Deskryptory i Generatory
### Skoroszyt główny

---

## Cele Laboratorium
Celem dzisiejszych zajęć jest opanowanie zaawansowanych konstrukcji języka Python, które są niezbędne do projektowania nowoczesnej architektury aplikacji.

### System Wspomagania AI (Tutor)
W trakcie rozwiązywania zadań możesz korzystać z pomocy dedykowanego tutora AI. System oferuje 6 poziomów wsparcia:
1. **Ogólna wskazówka**: Sugestia kierunku rozwiązania.
2. **Pseudokod**: Logiczny opis algorytmu.
3. **Mały fragment kodu**: Kluczowa linia lub konstrukcja.
4. **Częściowa implementacja**: Szkielet kodu do uzupełnienia.
5. **Szczegółowe wyjaśnienie**: Analiza mechanizmu działania.
6. **Pełne rozwiązanie**: Dostępne w sytuacjach ostatecznych.

---

## 1. Dekoratory

### DEMO: Dekorator @timer 
Stwórz dekorator @timer, który będzie mierzył i wyświetlał czas wykonania funkcji.

In [ ]:
import time
import functools

def timer(func):
    @functools.wraps(func) #sprawdzic
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
        print(f"Czas wykonania {func.__name__}: {end_time - start_time:.4f} s")
        return result
    return wrapper

@timer
def example_task():
    time.sleep(0.5)
    print("Zadanie zakończone.")

example_task()

Zadanie zakończone.
Czas wykonania example_task: 0.5006 s


### Zadanie 1: Liczba elementów listy
Stwórz dekorator, który będzie odpowiedzialny za wyświetlanie liczby elementów listy, jeśli jakakolwiek lista pojawi się w parametrach funkcji dekorowanej. 

**Protip:** użyj isinstance do sprawdzenia czy parametr jest listą. Pamiętaj o zachowaniu metadanych funkcji.

In [10]:
import functools

def show_list_length(func):
    @functools.wraps(func)
    def wrapper(*args,**kwargs):
        for arg in args:
            x = isinstance(arg, list)
            if x == True:
                print(len(arg))
        for kwarg in kwargs.values():
            y = isinstance(kwarg, list)
            if y == True:
                print(len(kwarg))
        result = func(*args,**kwargs)
        return result
    return wrapper    


@show_list_length
def process_data(data_list, name):
     print(f"Przetwarzanie {name}")
process_data(data_list=[1, 2, 3], name="test")


3
Przetwarzanie test


### Zadanie 2: Logowanie do pliku
Stwórz dekorator, który będzie zapisywał w pliku *.log nazwę funkcji dekorowanej, datę oraz długość wykonania. Nazwa pliku będzie podana jako argument dekoratora.

**Protip:** użyj biblioteki datetime. Pamiętaj o tym, żeby dekoratory przyjęły metadanych funkcji dekorującej.

In [ ]:
import functools
from datetime import datetime
import time

# TODO: Implementacja dekoratora z argumentem
def logger(filename):
    pass

--- 
## 2. Deskryptory

### DEMO: Walidator e-mail klasy Student
Stwórz deskryptor, który będzie działał jako walidator email klasy Student. Klasa Student zawiera pola imie, nazwisko i email. Deskryptor ten powinien sprawdzać poprawność danych wprowadzanych podczas tworzenia lub modyfikowania instancji Student.

In [ ]:
class EmailValidator:
    def __set_name__(self, owner, name):
        self.name = name

    def __set__(self, instance, value):
        if "@" not in value:
            raise ValueError(f"Błędny format adresu email: {value}")
        instance.__dict__[self.name] = value

class Student:
    email = EmailValidator()
    
    def __init__(self, imie, nazwisko, email):
        self.imie = imie
        self.nazwisko = nazwisko
        self.email = email

try:
    s = Student("Jan", "Kowalski", "jan.kowalski@wsei.edu.pl")
    print(f"Utworzono studenta: {s.email}")
    # s.email = "invalid_at_email" # Powinno rzucić błąd
except ValueError as e:
    print(e)

Utworzono studenta: jan.kowalski@wsei.edu.pl


### Zadanie 3: Rejestrowanie dostępu
Stwórz klasę Uzytkownik. Klasa powinna zawierać atrybuty imie i wiek. Opracuj deskryptor, który będzie rejestrował dostęp do tych atrybutów za pomocą logowania. Deskryptor powinien logować informacje o odczycie (__get__) oraz zapisie (__set__) wartości atrybutu.

In [1]:
class AccessLogger:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        
        print(self._format_log("Odczytano", self.name))
        
        return instance.__dict__.get(self.name)

    def __set__(self, instance, value):
        print(self._format_log("Zapisano", self.name, f"na wartość '{value}'"))
        
        instance.__dict__[self.name] = value

    def _format_log(self, action, name, extra=""):
        return f"[LOG] {action} atrybut '{name}'{extra}."


class Uzytkownik:
    imie = AccessLogger()
    wiek = AccessLogger()

    def __init__(self, imie, wiek):
        self.imie = imie
        self.wiek = wiek


if __name__ == "__main__":
    print("--- Tworzenie użytkownika (Zapis w __init__) ---")
    uzytkownik = Uzytkownik("Jan", 28)

    print("\n--- Odczyt atrybutów ---")
    print(f"Imię użytkownika: {uzytkownik.imie}")
    print(f"Wiek użytkownika: {uzytkownik.wiek}")

    print("\n--- Zmiana wartości (Zapis) ---")
    uzytkownik.imie = "Anna"
    uzytkownik.wiek = 30

    print("\n--- Ponowny odczyt po zmianie ---")
    print(f"Nowe imię: {uzytkownik.imie}")

--- Tworzenie użytkownika (Zapis w __init__) ---
[LOG] Zapisano atrybut 'imie'na wartość 'Jan'.
[LOG] Zapisano atrybut 'wiek'na wartość '28'.

--- Odczyt atrybutów ---
[LOG] Odczytano atrybut 'imie'.
Imię użytkownika: Jan
[LOG] Odczytano atrybut 'wiek'.
Wiek użytkownika: 28

--- Zmiana wartości (Zapis) ---
[LOG] Zapisano atrybut 'imie'na wartość 'Anna'.
[LOG] Zapisano atrybut 'wiek'na wartość '30'.

--- Ponowny odczyt po zmianie ---
[LOG] Odczytano atrybut 'imie'.
Nowe imię: Anna


--- 
## 3. Generatory i Iteratory

### DEMO: Generator Fibonacciego
Napisz klasę, która będzie implementowała generator ciągu Fibonacciego za pomocą metod magicznych __iter__() i __next__().

In [ ]:
class FibonacciGenerator:
    def __init__(self, limit):
        self.limit = limit
        self.a, self.b = 0, 1
        self.count = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.count >= self.limit:
            raise StopIteration
        
        result = self.a
        self.a, self.b = self.b, self.a + self.b
        self.count += 1
        return result

fib = FibonacciGenerator(10)
print(list(fib))

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


### Zadanie 4: Generator ciągu Collatza
Opracuj generator ciągu Collatza. Dla liczby naturalnej n, jeśli n jest parzyste, dziel przez 2; jeśli n jest nieparzyste, pomnóż przez 3 i dodaj 1, zaczynając od określonej liczby początkowej, aż do osiągnięcia wartości 1.

In [2]:
def collatz_generator(n):
    if not isinstance(n, int) or n <= 0:
        raise ValueError("Ciąg Collatza wymaga dodatniej liczby naturalnej!")

    yield f"Start z liczby: {n}"

    while n > 1:
        if n % 2 == 0:
            n = n // 2 
            yield f"Parzysta -> Dzielę przez 2: {n}"
        else:
            n = n * 3 + 1
            yield f"Nieparzysta -> 3n + 1: {n}"


if __name__ == "__main__":
    for status in collatz_generator(10):
        print(status)

Start z liczby: 10
Parzysta -> Dzielę przez 2: 5
Nieparzysta -> 3n + 1: 16
Parzysta -> Dzielę przez 2: 8
Parzysta -> Dzielę przez 2: 4
Parzysta -> Dzielę przez 2: 2
Parzysta -> Dzielę przez 2: 1


---

## Zadania do zrobienia w domu

Poniższe zadania stanowią rozszerzenie materiału i są przeznaczone dla osób chcących zgłębić temat zaawansowanych konstrukcji języka Python.

### Zadanie dodatkowe 1: Dekorator z autoryzacją
Stwórz dekorator `@require_role(role)`, który przyjmuje nazwę wymaganej roli jako argument. Dekorator powinien sprawdzać, czy w globalnym słowniku `current_user` klucz `role` jest zgodny z wymaganym. Jeśli nie, rzuć `PermissionError`.

In [3]:
from functools import wraps

current_user = {"username": "admin", "role": "superuser"}


def require_role(role):
    def decorator(func):
        @wraps(func) 
        def wrapper(*args, **kwargs):
            user_role = current_user.get("role")

            if user_role != role:
                raise PermissionError(
                    f"Brak dostępu! Wymagana rola: '{role}', "
                    f"Twoja aktualna rola: '{user_role}'."
                )

            return func(*args, **kwargs)

        return wrapper

    return decorator


@require_role("superuser")
def usun_baze_danych():
    return "Baza danych została pomyślnie usunięta."


@require_role("moderator")
def zablokuj_uzytkownika(user_id):
    return f"Użytkownik {user_id} został zablokowany."


if __name__ == "__main__":
    print(f"Aktualnie zalogowany jako: {current_user['username']} (Rola: {current_user['role']})\n")

    try:
        print("Próba wywołania usun_baze_danych()...")
        wynik = usun_baze_danych()
        print(f"Sukces: {wynik}\n")
    except PermissionError as e:
        print(f"Błąd: {e}\n")

    try:
        print("Próba wywołania zablokuj_uzytkownika(42)...")
        wynik = zablokuj_uzytkownika(42)
        print(f"Sukces: {wynik}\n")
    except PermissionError as e:
        print(f"Przechwycono oczekiwany wyjątek -> {e}\n")

    print("--- Zmiana użytkownika na zwykłego gracza ---")
    current_user["username"] = "Janek"
    current_user["role"] = "user"

    try:
        print("Próba wywołania usun_baze_danych() jako zwykły user...")
        usun_baze_danych()
    except PermissionError as e:
        print(f"Przechwycono oczekiwany wyjątek -> {e}")

Aktualnie zalogowany jako: admin (Rola: superuser)

Próba wywołania usun_baze_danych()...
Sukces: Baza danych została pomyślnie usunięta.

Próba wywołania zablokuj_uzytkownika(42)...
Przechwycono oczekiwany wyjątek -> Brak dostępu! Wymagana rola: 'moderator', Twoja aktualna rola: 'superuser'.

--- Zmiana użytkownika na zwykłego gracza ---
Próba wywołania usun_baze_danych() jako zwykły user...
Przechwycono oczekiwany wyjątek -> Brak dostępu! Wymagana rola: 'superuser', Twoja aktualna rola: 'user'.


### Zadanie dodatkowe 2: Deskryptor z walidacją typu
Stwórz deskryptor `Typed`, który przyjmuje typ danych (np. `int`, `str`) w konstruktorze. Deskryptor powinien upewnić się, że zapisywana wartość jest tego typu. Jeśli nie, rzuć `TypeError`.

In [4]:
class Typed:
    def __init__(self, expected_type):
        self.expected_type = expected_type

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__.get(self.name)

    def __set__(self, instance, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(
                f"Nieprawidłowy typ dla atrybutu '{self.name}'! "
                f"Oczekiwano: {self.expected_type.__name__}, "
                f"otrzymano: {type(value).__name__}."
            )
        
        instance.__dict__[self.name] = value


class Produkt:
    nazwa = Typed(str)
    cena = Typed(float)
    ilosc = Typed(int)

    def __init__(self, nazwa, cena, ilosc):
        self.nazwa = nazwa
        self.cena = cena
        self.ilosc = ilosc


if __name__ == "__main__":
    print("--- Test 1: Tworzenie obiektu z poprawnymi danymi ---")
    try:
        prod = Produkt("Laptop", 3499.99, 5)
        print(f"Pomyślnie utworzono produkt: {prod.nazwa}, Cena: {prod.cena} zł, Ilość: {prod.ilosc} szt.\n")
    except TypeError as e:
        print(f"Błąd (nieoczekiwany): {e}\n")

    print("--- Test 2: Próba przypisania złego typu do istniejącego obiektu ---")
    try:
        print("Próba zmiany ilości sztuk na tekst 'dziesięć'...")
        prod.ilosc = "dziesięć" 
    except TypeError as e:
        print(f"Przechwycono oczekiwany błąd -> {e}\n")

    print("--- Test 3: Próba utworzenia obiektu ze złymi danymi na starcie ---")
    try:
        print("Próba stworzenia produktu, gdzie cena to liczba całkowita (int) zamiast float...")
        zly_prod = Produkt("Telefon", 2500, 1) 
    except TypeError as e:
        print(f"Przechwycono oczekiwany błąd -> {e}")

--- Test 1: Tworzenie obiektu z poprawnymi danymi ---
Pomyślnie utworzono produkt: Laptop, Cena: 3499.99 zł, Ilość: 5 szt.

--- Test 2: Próba przypisania złego typu do istniejącego obiektu ---
Próba zmiany ilości sztuk na tekst 'dziesięć'...
Przechwycono oczekiwany błąd -> Nieprawidłowy typ dla atrybutu 'ilosc'! Oczekiwano: int, otrzymano: str.

--- Test 3: Próba utworzenia obiektu ze złymi danymi na starcie ---
Próba stworzenia produktu, gdzie cena to liczba całkowita (int) zamiast float...
Przechwycono oczekiwany błąd -> Nieprawidłowy typ dla atrybutu 'cena'! Oczekiwano: float, otrzymano: int.


### Zadanie dodatkowe 3: Nieskończony generator liczb pierwszych
Opracuj generator `prime_generator`, który zwraca kolejne liczby pierwsze. Następnie użyj wyrażenia generatorowego, aby stworzyć iterator zwracający tylko te liczby pierwsze, które kończą się cyfrą 7.

In [5]:
import math

def is_prime(num):
    """Funkcja pomocnicza sprawdzająca, czy liczba jest pierwsza."""
    if num < 2:
        return False
    if num == 2:
        return True
    if num % 2 == 0:
        return False
    for i in range(3, int(math.sqrt(num)) + 1, 2):
        if num % i == 0:
            return False
    return True


def prime_generator():
    """Nieskończony generator kolejnych liczb pierwszych."""
    current = 2
    while True:
        if is_prime(current):
            yield current
        current += 1


wszystkie_pierwsze = prime_generator()

primes_ending_in_7 = (p for p in wszystkie_pierwsze if p % 10 == 7)

if __name__ == "__main__":
    print("--- Test 1: Pierwsze 10 standardowych liczb pierwszych ---")
    test_gen = prime_generator()
    for _ in range(10):
        print(next(test_gen), end=" ")
    print("\n")

    print("--- Test 2: Pierwsze 10 liczb pierwszych kończących się na 7 ---")

    for _ in range(10):
        print(next(primes_ending_in_7), end=" ")
    print()

--- Test 1: Pierwsze 10 standardowych liczb pierwszych ---
2 3 5 7 11 13 17 19 23 29 

--- Test 2: Pierwsze 10 liczb pierwszych kończących się na 7 ---
7 17 37 47 67 97 107 127 137 157 
